## Caso práctico: Transformación e integración de datos de ventas

La empresa **RetailNova Perú** está modernizando su área de datos. Hoy la información de ventas se encuentra dispersa en **tres fuentes distintas**, gestionadas por equipos diferentes:

1. `ventas.csv` — transacciones diarias registradas en caja (≈ 500 registros).
2. `clientes.csv` — base de clientes del CRM (≈ 320 registros).
3. `productos.csv` — catálogo de productos del almacén (≈ 300 registros).

El problema: cada archivo fue capturado por un equipo distinto y contiene **problemas de calidad de datos** (valores nulos, registros duplicados, formatos inconsistentes). Antes de poder analizar o construir métricas, es necesario **limpiar, integrar y transformar** la información en un único dataset confiable.

### Objetivo general

Construir un **pipeline de procesamiento de datos** que permita:

1. **Cargar** las tres fuentes de datos.
2. **Limpiar** los datos (nulos, duplicados e inconsistencias).
3. **Integrar** las tablas mediante `merge` y `join`.
4. **Pivotear** la información para el análisis (`pivot`, `melt`, `stack`, `unstack`).
5. **Derivar** métricas de negocio (KPI).
6. **Exportar** el dataset final listo para análisis.

### Pipeline de procesamiento

```mermaid
flowchart LR
    A[Cargar Datos<br/>3 archivos CSV] --> B[Limpiar<br/>Nulos y duplicados]
    B --> C[Integrar<br/>merge / join]
    C --> D[Pivotear<br/>pivot / melt / stack]
    D --> E[Derivar<br/>Métricas KPI]
    E --> F[Exportar<br/>Dataset final]
```

In [ ]:
# 1. Cargar librerías necesarias
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# 2. Cargar las tres fuentes de datos
ventas = pd.read_csv("./data/ventas.csv", encoding="utf-8")
clientes = pd.read_csv("./data/clientes.csv", encoding="utf-8")
productos = pd.read_csv("./data/productos.csv", encoding="utf-8")

print("ventas   :", ventas.shape)
print("clientes :", clientes.shape)
print("productos:", productos.shape)

In [ ]:
# 3. Vista previa de cada tabla
display(ventas.head(3))
display(clientes.head(3))
display(productos.head(3))

In [ ]:
# 4. Información general de los datasets
ventas.info()
print("\n---\n")
clientes.info()
print("\n---\n")
productos.info()

### Observaciones iniciales

- `ventas`: columnas `id_cliente`, `id_producto`, `cantidad` y `precio_unitario` tienen valores nulos; las fechas mezclan formatos; `sucursal` y `metodo_pago` presentan variaciones de escritura.
- `clientes`: nulos en `ciudad` y `edad`; edades fuera de rango; `segmento` con distinta capitalización.
- `productos`: nulos en `categoria` y `precio_lista`; categorías con espacios y minúsculas.

In [ ]:
# 5. Conteo de valores nulos por tabla
print("-- Nulos en ventas:--\n", ventas.isna().sum())
print("\n-- Nulos en clientes:--\n", clientes.isna().sum())
print("\n-- Nulos en productos:--\n", productos.isna().sum())

In [ ]:
# 6. Limpieza de valores nulos

# 6.1 ventas: eliminar filas sin clave foránea ni medidas
ventas_limpio = ventas.dropna(
    subset=["id_cliente", "id_producto", "cantidad", "precio_unitario"]
).copy()
# completar método de pago faltante
ventas_limpio["metodo_pago"] = ventas_limpio["metodo_pago"].fillna("SIN REGISTRO")

# 6.2 clientes: completar ciudad y edad
clientes_limpio = clientes.copy()
clientes_limpio["ciudad"] = clientes_limpio["ciudad"].fillna("SIN REGISTRO")
clientes_limpio["edad"] = clientes_limpio["edad"].fillna(clientes_limpio["edad"].median())

# 6.3 productos: completar categoría y precio de lista
productos_limpio = productos.copy()
productos_limpio["categoria"] = productos_limpio["categoria"].fillna("SIN CATEGORIA")
productos_limpio["precio_lista"] = productos_limpio["precio_lista"].fillna(
    productos_limpio["precio_lista"].median()
)

In [ ]:
# 7. Análisis y eliminación de registros duplicados
print("Duplicados en ventas   :", ventas_limpio.duplicated().sum())
print("Duplicados en clientes :", clientes_limpio.duplicated(subset=["id_cliente"]).sum())
print("Duplicados en productos:", productos_limpio.duplicated(subset=["id_producto"]).sum())

ventas_limpio = ventas_limpio.drop_duplicates()
clientes_limpio = clientes_limpio.drop_duplicates(subset=["id_cliente"])
productos_limpio = productos_limpio.drop_duplicates(subset=["id_producto"])

In [ ]:
# 8. Consistencia: normalizar textos (mayúsculas y sin espacios)
ventas_limpio["sucursal"] = ventas_limpio["sucursal"].str.upper().str.strip()
ventas_limpio["metodo_pago"] = ventas_limpio["metodo_pago"].str.upper().str.strip()

clientes_limpio["ciudad"] = clientes_limpio["ciudad"].str.upper().str.strip()
clientes_limpio["segmento"] = clientes_limpio["segmento"].str.upper().str.strip()

productos_limpio["categoria"] = productos_limpio["categoria"].str.upper().str.strip()

In [ ]:
# 9. Consistencia: normalizar fechas (formatos mixtos) a datetime
ventas_limpio["fecha"] = pd.to_datetime(
    ventas_limpio["fecha"], errors="coerce", format="mixed"
)
print("Fechas inválidas (NaT):", ventas_limpio["fecha"].isna().sum())
ventas_limpio = ventas_limpio.dropna(subset=["fecha"])

In [ ]:
# 10. Consistencia: filtrar valores fuera de rango
# Cantidades negativas o excesivas en ventas
ventas_limpio = ventas_limpio[
    (ventas_limpio["cantidad"] > 0) & (ventas_limpio["cantidad"] <= 50)
]

# Edades irreales en clientes
clientes_limpio = clientes_limpio[
    (clientes_limpio["edad"] > 0) & (clientes_limpio["edad"] <= 100)
]

# Precios no positivos en productos
productos_limpio = productos_limpio[productos_limpio["precio_lista"] > 0]

In [ ]:
# 11. Consistencia referencial: conservar solo claves que existen en las dimensiones
ventas_limpio = ventas_limpio[ventas_limpio["id_cliente"].isin(clientes_limpio["id_cliente"])]
ventas_limpio = ventas_limpio[ventas_limpio["id_producto"].isin(productos_limpio["id_producto"])]

print("Registros finales en ventas:", ventas_limpio.shape[0])

In [ ]:
# 12. Verificación final de la limpieza
print("Nulos restantes en ventas:\n", ventas_limpio.isna().sum())
print("\nDuplicados restantes:", ventas_limpio.duplicated().sum())

### Integración de datos

Con las tres tablas limpias, las integramos en un único dataframe analítico:

- **`merge`**: combina columnas de dos tablas a partir de una o varias claves comunes.
- **`join`**: une dos tablas usando sus índices (o el índice de una y una columna de la otra).

In [ ]:
# 13. Integrar con merge: ventas + productos (inner join por id_producto)
df = ventas_limpio.merge(productos_limpio, on="id_producto", how="inner")

# merge: agregar datos del cliente (inner join por id_cliente)
df = df.merge(clientes_limpio, on="id_cliente", how="inner")

print("Dataset integrado:", df.shape)
df.head()

In [ ]:
# 14. Alternativa con join (usa el índice de la tabla derecha)
# Convertimos id_cliente en el índice de clientes y unimos por la izquierda
df_join = ventas_limpio.join(
    clientes_limpio.set_index("id_cliente"), on="id_cliente", how="left"
)
print("Resultado del join (izquierda):", df_join.shape)
df_join[["id_venta", "id_cliente", "nombre", "ciudad", "segmento"]].head()

In [ ]:
# 15. Crear la medida clave para el análisis: importe de la venta
df["importe"] = df["cantidad"] * df["precio_unitario"]
df["mes"] = df["fecha"].dt.to_period("M")
df.head()

### Pivoteo de tablas

Transformamos la tabla larga (una fila por venta) en tablas resumen para el análisis:

- **`pivot_table` / `pivot`**: tabla larga → tabla ancha (resumen por filas y columnas).
- **`melt`**: tabla ancha → tabla larga (unpivot).
- **`stack` / `unstack`**: mover niveles entre filas y columnas en un `MultiIndex`.

In [ ]:
# 16. Pivot table: importe total por sucursal (filas) y categoría (columnas)
pivot_ingresos = df.pivot_table(
    index="sucursal", columns="categoria",
    values="importe", aggfunc="sum", fill_value=0
)
pivot_ingresos.round(2)

In [ ]:
# 17. Pivot (requiere claves únicas): agrupamos primero para asegurar unicidad
base = df.groupby(["sucursal", "categoria"], as_index=False)["importe"].sum()
pivot_estricto = base.pivot(
    index="sucursal", columns="categoria", values="importe"
).fillna(0)
pivot_estricto.round(2)

In [ ]:
# 18. Melt (unpivot): devolver la tabla ancha a formato largo
tabla_larga = pivot_ingresos.reset_index().melt(
    id_vars="sucursal", var_name="categoria", value_name="importe"
)
tabla_larga.head(10)

In [ ]:
# 19. Stack: convertir columnas (categorías) en filas -> Series con MultiIndex
apilado = pivot_estricto.stack()
print("Resultado de stack (MultiIndex):")
apilado.head(10)

In [ ]:
# 20. Unstack: devolver el MultiIndex a formato ancho
desapilado = apilado.unstack(fill_value=0)
desapilado.round(2)

In [ ]:
# 21. Melt sobre ventas mensuales por sucursal (ancho -> largo)
ventas_mensuales = df.pivot_table(
    index="mes", columns="sucursal",
    values="importe", aggfunc="sum", fill_value=0
)
ventas_mensuales_long = ventas_mensuales.reset_index().melt(
    id_vars="mes", var_name="sucursal", value_name="importe"
)
ventas_mensuales_long.head()

### Métricas derivadas (KPI)

Con el dataset integrado y limpio calculamos los indicadores que la gerencia necesita para tomar decisiones.

In [ ]:
# 22. Calcular métricas de negocio
total_ingresos = df["importe"].sum()
ticket_promedio = df["importe"].mean()
num_transacciones = df["id_venta"].nunique()
clientes_unicos = df["id_cliente"].nunique()
productos_vendidos = df["id_producto"].nunique()

kpis = pd.DataFrame({
    "Métrica": ["Ingresos totales", "Ticket promedio", "N° transacciones",
                "Clientes únicos", "Productos vendidos"],
    "Valor": [f"{total_ingresos:,.2f}", f"{ticket_promedio:,.2f}",
              num_transacciones, clientes_unicos, productos_vendidos]
})
kpis

In [ ]:
# 23. Top 5 productos por ingresos
top_productos = df.groupby("nombre_producto")["importe"].sum().nlargest(5)
top_productos.plot.bar(
    figsize=(10, 5), title="Top 5 productos por ingresos",
    ylabel="Ingresos", rot=45, legend=False, fontsize=12
)

In [ ]:
# 24. Exportar el dataset final y la tabla pivoteada
df.to_csv("./exports/dataset_ventas_final.csv", index=False, encoding="utf-8")
pivot_ingresos.to_csv("./exports/pivot_ingresos_sucursal_categoria.csv", encoding="utf-8")

print("Dataset final exportado a ./exports/dataset_ventas_final.csv")

### Conclusiones

- La **limpieza** eliminó nulos, duplicados y valores fuera de rango, dejando un dataset íntegro.
- La **normalización de textos** evitó que valores como `"LIMA"` y `" lima"` se contaran como categorías distintas.
- Con **`merge`** integramos las tres fuentes en un único dataframe analítico; con **`join`** vimos una alternativa basada en índices.
- Con **`pivot_table`/`pivot`** resumimos ingresos por sucursal y categoría; con **`melt`** y **`stack`/`unstack`** revertimos el proceso entre formato ancho y largo.
- Las **métricas** permiten a la gerencia medir ingresos, ticket promedio y desempeño por producto.